# sre-gym — Training pipeline sanity run

Purpose: verify the Colab+Unsloth+TRL+wandb pipeline compiles and runs end-to-end on an A100 *before* the hackathon. This notebook is not meant to train anything useful. It runs 200 SFT steps on a tiny hand-made dataset and saves a checkpoint.

What a successful run looks like:
1. All deps install without version conflicts
2. `Qwen3.5-4B-Instruct` (or `Qwen3-4B-Instruct` fallback) loads in 4-bit via Unsloth
3. 200 steps of LoRA SFT run without OOM on A100 40GB
4. `wandb` logs show loss decreasing
5. Checkpoint is saved to `/content/sanity_ckpt/`

Friday work: real dataset (2000+ Claude-driven trajectories), 2000+ SFT steps, then GRPO.

## 0. Colab runtime sanity

In [ ]:
!nvidia-smi
!python -c 'import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())'

## 1. Install deps

In [ ]:
%%bash
# Unsloth's Colab install idiom (handles torch/xformers version pinning):
pip install -q --upgrade pip
pip install -q "unsloth[colab-new]>=2025.12,<2026.06"
pip install -q "unsloth_zoo>=2025.12,<2026.06"
pip install -q "trl>=0.12,<0.16" "transformers>=4.48,<4.60" "peft>=0.14,<0.20" "accelerate>=1.2,<2.0"
pip install -q "datasets>=3.0,<4.0" "wandb>=0.18,<1.0" "bitsandbytes>=0.45" httpx

## 2. Config

If Qwen3.5 4B fails to load, swap `MODEL_NAME` to the Qwen3 4B fallback — no other change needed.

In [ ]:
import os

# Primary target (user-selected).
MODEL_NAME = "unsloth/Qwen3.5-4B-Instruct-bnb-4bit"
# Fallback if Unsloth can't load Qwen3.5 on Colab tonight.
FALLBACK_MODEL_NAME = "unsloth/Qwen3-4B-Instruct-bnb-4bit"

MAX_SEQ_LENGTH = 4096
LORA_R = 32
LORA_ALPHA = 32
LEARNING_RATE = 2e-4
NUM_STEPS = 200
BATCH_SIZE = 2
GRAD_ACCUM = 4
OUT_DIR = "/content/sanity_ckpt"

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "sre-gym-sanity")
WANDB_RUN_NAME = os.environ.get("WANDB_RUN_NAME", "qwen35-4b-sft-toy-200")

os.environ.setdefault("WANDB_MODE", "online")  # flip to "offline" if no wandb login
print(f"Primary model: {MODEL_NAME}")

## 3. Load model via Unsloth (with fallback)

In [ ]:
from unsloth import FastLanguageModel
import torch

model = None
tokenizer = None
errors = []

for candidate in (MODEL_NAME, FALLBACK_MODEL_NAME):
    try:
        print(f"Attempting to load: {candidate}")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=candidate,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,  # let Unsloth pick
            load_in_4bit=True,
        )
        MODEL_NAME = candidate
        print(f"Loaded {candidate} ok")
        break
    except Exception as exc:
        errors.append((candidate, repr(exc)))
        print(f"Load failed for {candidate}: {exc}")

if model is None:
    raise RuntimeError(
        "Both Qwen3.5 4B and Qwen3 4B failed to load via Unsloth. "
        "Investigate Unsloth version mismatch before Friday. Errors: " + str(errors)
    )

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 4. Toy training dataset (hand-made, 10 examples)

These are derived from the 3 deterministic baseline trajectories. Purpose: exercise the tokenize+forward+backward+optimizer path. Not intended to generalize.

In [ ]:
import json
from datasets import Dataset

SYSTEM = 'You are an SRE agent. Respond with one UnifiedIncidentAction JSON object on each turn.'

TOY_EXAMPLES = [
    ("worker_deploy_cascade tick 1 — DB crashed, worker degraded, recent worker deploy",
     '{"action_type":"query_deploys","service":"worker"}'),
    ("worker_deploy_cascade tick 2 — saw worker@2026.04.23-bad 12m ago",
     '{"action_type":"query_logs","service":"worker"}'),
    ("worker_deploy_cascade tick 3 — confirmed worker-originated harm",
     '{"action_type":"rollback_deploy","service":"worker"}'),
    ("worker_deploy_cascade tick 4 — worker healthy, DB still crashed",
     '{"action_type":"restart_service","service":"database"}'),
    ("worker_deploy_cascade tick 5 — all services healthy, checks pending",
     '{"action_type":"run_check","check_name":"end_to_end"}'),
    ("db_config_rollout tick 1 — db degraded, worker decoy, pool-acquire errors",
     '{"action_type":"query_deploys","service":"database"}'),
    ("db_config_rollout tick 2 — saw db@2026.04.24-cfg lowering pool to 12",
     '{"action_type":"rollback_deploy","service":"database"}'),
    ("gateway_auth_rollout tick 1 — gateway 40% 401s, auth rollout 9m ago",
     '{"action_type":"query_deploys","service":"api-gateway"}'),
    ("gateway_auth_rollout tick 2 — confirmed gateway@2026.04.24-auth is cause",
     '{"action_type":"rollback_deploy","service":"api-gateway"}'),
    ("gateway_auth_rollout tick 3 — gateway healthy, verify end-to-end",
     '{"action_type":"run_check","check_name":"end_to_end"}'),
]

def _format(example):
    prompt, action = example
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": action},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

raw = [_format(ex) for ex in TOY_EXAMPLES]
dataset = Dataset.from_list(raw)
print(f"toy dataset: {len(dataset)} rows")
print("sample text (first 400 chars):")
print(dataset[0]['text'][:400])

## 5. SFT training — 200 steps

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=10,
    max_steps=NUM_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=cfg,
)

trainer_stats = trainer.train()
print(trainer_stats)

## 6. Save LoRA adapter + sanity-check inference

In [ ]:
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_prompt = 'worker_deploy_cascade tick 1 — DB crashed, worker degraded, recent worker deploy'
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": test_prompt},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=64, temperature=0.0, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))

## Verification checklist

- [ ] Cell 3 loaded a model without OOM or import errors
- [ ] Cell 4 produced a chat-formatted dataset (no tokenizer errors)
- [ ] Cell 5 ran 200 steps, wandb logged a decreasing loss curve
- [ ] Cell 6 generated a JSON-ish action for the test prompt
- [ ] `/content/sanity_ckpt/adapter_model.safetensors` exists

If any box is unchecked, debug tonight — do not enter Friday with an unknown failure mode.